# Using the milos SDK

`milos` exposes the same shape as the [Claude Agent SDK](https://code.claude.com/docs/en/agent-sdk):
`query()` for one turn, `MilosClient` for a conversation, and the Agent SDK's own message types
(`AssistantMessage`, `UserMessage`, `SystemMessage`, `ResultMessage`, `TextBlock`, `ToolUseBlock`, `ToolResultBlock`).
Code written against `claude_agent_sdk` reads the same here.

The difference is where the agent runs. With the Agent SDK the model and its tools run in your process.
With milos they run on the platform: the agent's definition decides which tools it may use, tools it may
not use without a person park the session, and everything is journaled and audited. Your script never holds
a model key, a tool credential or a security decision it did not ask for.

Settings come from `.env` at the repository root (copy `.env.example`; the file is ignored by git).
Run the notebook with the project environment: `uv run --with jupyter jupyter lab docs/sdk.ipynb`.

## 1. Identity

The API sits behind Identity-Aware Proxy (IAP). `MilosOptions` takes the API URL and an identity token,
or reads `MILOS_API_URL` and mints a token with `gcloud auth print-identity-token` when they are omitted.

The dev environment uses the Google-managed OAuth client, which admits people in a browser but not from code,
so this notebook acts as the two service accounts created in `infra/envs/dev/operators.tf`: an **operator**
who starts sessions, and an **approver** who decides tool calls. Anyone in the users group may sign a JWT
for either; IAP accepts it with the service URL plus `/*` as audience.

In [ ]:
import json
import os
import subprocess
import tempfile
import time
from pathlib import Path

for line in (Path("..") / ".env").read_text().splitlines():
    if line.strip() and not line.startswith("#") and "=" in line:
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip())

API_URL = os.environ["MILOS_API_URL"]
RUNTIME_PROJECT = os.environ["MILOS_RUNTIME_PROJECT"]
OPERATOR = f"milos-operator@{RUNTIME_PROJECT}.iam.gserviceaccount.com"
APPROVER = f"milos-approver@{RUNTIME_PROJECT}.iam.gserviceaccount.com"


def iap_token(service_account: str, *, ttl: int = 3600) -> str:
    """Sign a JWT as the service account; IAP accepts it as that identity."""
    now = int(time.time())
    claims = {
        "iss": service_account, "sub": service_account, "email": service_account,
        "aud": API_URL.rstrip("/") + "/*", "iat": now, "exp": now + ttl,
    }
    with tempfile.TemporaryDirectory() as tmp:
        claims_path, out_path = Path(tmp) / "claims.json", Path(tmp) / "token.jwt"
        claims_path.write_text(json.dumps(claims))
        subprocess.run(
            ["gcloud", "iam", "service-accounts", "sign-jwt", f"--iam-account={service_account}", claims_path, out_path],
            check=True, capture_output=True,
        )
        return out_path.read_text().strip()


operator_token, approver_token = iap_token(OPERATOR), iap_token(APPROVER)
API_URL

## 2. One turn with `query`

As in the Agent SDK, `query` takes a prompt and options and yields messages until the turn ends.
The options name an agent instead of a model and tools: those live in the agent's definition on the platform.

In [ ]:
from milos import AssistantMessage, MilosOptions, ResultMessage, TextBlock, ToolResultBlock, ToolUseBlock, UserMessage, query

options = MilosOptions(
    agent="analyst",
    api_url=API_URL,
    token=operator_token,
    approvers=[APPROVER],
)


def show(message):
    match message:
        case AssistantMessage(content=[TextBlock(text=text)]):
            print("agent:", text)
        case AssistantMessage(content=[ToolUseBlock(name=name, input=args)]):
            print("tool:", name, args)
        case UserMessage(content=[ToolResultBlock(content=summary, is_error=err)]):
            print("  ->", "failed" if err else "ok", str(summary)[:120])
        case UserMessage(content=str() as text):
            print("you:", text)
        case ResultMessage(subtype=subtype, num_turns=turns, total_cost_usd=cost):
            print(f"[{subtype}] turns={turns} cost=${cost or 0:.4f}")
        case _:
            print(f"[{message.subtype}] {message.data}")


async for message in query("List the files you can see and summarise the most recent week.", options):
    show(message)

## 3. A conversation with `MilosClient`

`MilosClient` mirrors `ClaudeSDKClient`: `query` sends a message, `receive_response` yields the messages
of that turn, `interrupt` stops the runner, and the client is an async context manager. The session keeps
its context between turns. Unlike the Agent SDK, the session outlives the client: `disconnect` closes
the connection, `terminate` ends the session.

In [ ]:
from milos import MilosClient

async with MilosClient(options) as client:
    await client.query("Summarise the most recent week in three bullets.")
    async for message in client.receive_response():
        show(message)

    await client.query("Now compare it with the week before.")
    async for message in client.receive_response():
        show(message)

    print("session:", client.session_id)
    await client.terminate()

## 4. Tool approval with `can_use_tool`

In the Agent SDK, `can_use_tool` is called when a tool needs permission. Here it is called when the
platform parks the session on a tool the definition does not pre-approve. Two things stay with the platform:

- The decision is made **as the approver**, never as the operator. Pass `approver_token`; the API refuses
  an approver who is also the operator.
- The decision is journaled and written to the audit log before the tool may run, whoever made it.

Return `PermissionResultAllow()` or `PermissionResultDeny(message=...)`, as in the Agent SDK.

In [ ]:
import shlex

from milos import PermissionResultAllow, PermissionResultDeny, ToolPermissionContext

READ_ONLY = {"ls", "cat", "head", "tail", "wc", "awk", "sort", "uniq", "pwd", "find", "grep"}


async def policy(tool_name: str, args: dict, context: ToolPermissionContext):
    print("deciding", context.tool_use_id, tool_name, args)
    if tool_name == "Bash":
        words = shlex.split(str(args.get("command", "")))
        if words and words[0] in READ_ONLY:
            return PermissionResultAllow()
        return PermissionResultDeny(message="only read-only commands")
    return PermissionResultDeny(message="no policy for this tool")


gated = MilosOptions(
    agent="analyst",
    api_url=API_URL,
    token=operator_token,
    approvers=[APPROVER],
    approver_token=approver_token,
    can_use_tool=policy,
)

async for message in query("Count the lines in every file with wc.", gated):
    show(message)

Without `can_use_tool`, a parked session ends the turn with a `ResultMessage` whose `subtype` is
`requires_action`; a person decides it in the UI or another script calls `Client.confirm` as the approver.

## 5. Unattended, idempotent runs

Schedulers retry. `client_request_id` makes the first `query` retry-safe: the same id with the same prompt
returns the existing session instead of starting a second one, and the same id with a different prompt
is refused with status 409.

In [ ]:
from datetime import date

from milos import ApiError

week = date.today().isocalendar()
weekly = MilosOptions(
    agent="analyst",
    api_url=API_URL,
    token=operator_token,
    client_request_id=f"weekly-report-{week.year}-W{week.week:02d}",
)

try:
    async for message in query("Write the weekly report for the latest week.", weekly):
        if isinstance(message, ResultMessage):
            print(message.subtype, message.result)
except ApiError as e:
    print(e.status, e.detail)

## Reference

| milos | Agent SDK | Notes |
| --- | --- | --- |
| `query(prompt, options)` | `query(prompt=, options=)` | One turn of a new session |
| `MilosOptions(agent, ...)` | `ClaudeAgentOptions(model, allowed_tools, ...)` | Model and tools come from the agent's definition |
| `MilosClient(options)` | `ClaudeSDKClient(options)` | Async context manager |
| `.query(prompt)` / `.receive_response()` / `.receive_messages()` / `.interrupt()` | same | |
| `.disconnect()` | same | The session stays on the platform |
| `.terminate()` | none | Ends the session |
| `.session_id` | `ResultMessage.session_id` | |
| `can_use_tool` + `approver_token` | `can_use_tool` | Decides as the approver; journaled and audited |
| `client_request_id` | none | Retry-safe session creation |

`ResultMessage.subtype` is `success`, `requires_action`, `error_max_budget`, `stopped`,
`error_during_execution` or `terminated`; `stop_reason` carries the platform's own value.
The lower-level `milos.Client` remains for anything the SDK shape does not cover, such as listing sessions
or reading a journal by sequence number.